In [ ]:
#import pandas as pd

# Az általad megadott link alapján a dokumentum azonosítója és a munkalap (gid) azonosítója
#sheet_id = "1i6EGSZa8Kx7SZKGgbDSbcBnq4ru5NZ4A"
#gid = "651728427"

# A link átalakítása úgy, hogy a pandas le tudja tölteni CSV formátumban
#url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={gid}"

#print("Adatok letöltése a Google Sheets-ből...")

# Beolvasás
#df_excel = pd.read_csv(url)

#print(f"KÉSZ! Sikeresen betöltve {len(df_excel)} sor.")
#print("Íme az első 5 sor:")
#display(df_excel.head())

Adatok letöltése a Google Sheets-ből...
KÉSZ! Sikeresen betöltve 687 sor.
Íme az első 5 sor:


,Bus stop,Bus stop (short),Vehicle type,Coordinates,Covered,Lightning,Wheelchair accessible,Bus bay available,Served by depot,Track type,Platform type,Platform name (short)
0,Leiningen utca,Leiningen utca,"Busz, E-Busz","21.629341, 47.514369",No,Yes,Yes,No,No,Non-fixed route,Departing/Arriving bus,"Mikepércsi út, Ozmán utca felé"
1,Leiningen utca,Leiningen utca,"Busz, E-Busz","21.629407, 47.5154",Yes,Yes,Yes,Yes,No,Non-fixed route,Departing/Arriving bus,"Mikepércsi út, Nagyállomás felé"
2,Somlyai utca,Somlyai utca,"Busz, E-Busz","21.630751, 47.504264",Yes,Yes,Yes,Yes,No,Non-fixed route,Departing/Arriving bus,"Mikepércsi út, Nagyállomás felé"
3,Somlyai utca,Somlyai utca,"Busz, E-Busz","21.630731, 47.502957",Yes,Yes,Yes,Yes,No,Non-fixed route,Departing/Arriving bus,"Mikepércsi út, Ozmán utca felé"
4,Repülőtéri út,Repülőtéri út,"Busz, E-Busz","21.616977, 47.502953",No,Yes,Yes,No,No,Non-fixed route,Departing/Arriving bus,"Dobogó utca, Airport Debrecen felé"


In [ ]:
import pandas as pd
import requests
import time

# Ide töltsd be a saját táblázatodat, amiben a 'szelesseg' és 'hosszusag' oszlopok vannak
# df_stops = pd.read_csv('a_te_fajlod.csv') 

# Példa struktúra a teszteléshez, ha még nem töltötted be:
if 'df_stops' not in locals():
    df_stops = pd.DataFrame({
        'nev': ['Segner tér', 'Nagyállomás'],
        'szelesseg': [47.5335, 47.5219],
        'hosszusag': [21.6212, 21.6294]
    })

In [ ]:
WEATHER_API_KEY = "IDE_IRD_AZ_OPENWEATHER_KULCSODAT"

def get_weather(lat, lon):
    if WEATHER_API_KEY == "IDE_IRD_AZ_OPENWEATHER_KULCSODAT":
        return 24.5, 52.0
    
    url = f"https://api.openweathermap.org/data/2.5/weather?lat={lat}&lon={lon}&appid={WEATHER_API_KEY}&units=metric"
    try:
        res = requests.get(url).json()
        temp = res['main']['temp']
        humidity = res['main']['humidity']
        return temp, humidity
    except:
        return None, None

temps = []
humidities = []

for idx, row in df_stops.iterrows():
    t, h = get_weather(row['szelesseg'], row['hosszusag'])
    temps.append(t)
    humidities.append(h)
    time.sleep(1)

df_stops['homerseklet'] = temps
df_stops['paratartalom'] = humidities

In [ ]:
def get_osm_features(lat, lon):
    url = "https://overpass-api.de/api/interpreter"
    query = f"""
    [out:json];
    (
      node["natural"="tree"](around:30,{lat},{lon});
      node["amenity"="bench"](around:30,{lat},{lon});
    );
    out body;
    """
    headers = {'User-Agent': 'DebrecenDataProject/1.0'}
    try:
        res = requests.post(url, data={'data': query}, headers=headers).json()
        elements = res.get('elements', [])
        trees = sum(1 for e in elements if e.get('tags', {}).get('natural') == 'tree')
        benches = sum(1 for e in elements if e.get('tags', {}).get('amenity') == 'bench')
        return trees, benches
    except:
        return 0, 0

tree_counts = []
bench_counts = []

for idx, row in df_stops.iterrows():
    trees, benches = get_osm_features(row['szelesseg'], row['hosszusag'])
    tree_counts.append(trees)
    bench_counts.append(benches)
    time.sleep(2)

df_stops['zold_novenyzet_szam'] = tree_counts
df_stops['padok_szama'] = bench_counts
df_stops['ferohely_becsles'] = df_stops['padok_szama'] * 3

In [ ]:
GOOGLE_API_KEY = "IDE_IRD_A_GOOGLE_MAPS_KULCSODAT"

def get_complaints(lat, lon, name):
    if GOOGLE_API_KEY == "IDE_IRD_A_GOOGLE_MAPS_KULCSODAT":
        return "Nincs API kulcs megadva (Minta panasz: Kevés az árnyék nyáron.)"
    
    search_url = f"https://maps.googleapis.com/maps/api/place/nearbysearch/json?location={lat},{lon}&radius=30&keyword={name}&key={GOOGLE_API_KEY}"
    try:
        place_res = requests.get(search_url).json()
        if not place_res.get('results'):
            return "Nem található értékelés a Google Maps-en."
        
        place_id = place_res['results'][0]['place_id']
        details_url = f"https://maps.googleapis.com/maps/api/place/details/json?place_id={place_id}&fields=reviews&key={GOOGLE_API_KEY}"
        details_res = requests.get(details_url).json()
        
        reviews = details_res.get('result', {}).get('reviews', [])
        text_reviews = [r['text'] for r in reviews if r.get('text')]
        
        if text_reviews:
            return " | ".join(text_reviews[:3])
        return "Nincsenek szöveges panaszok."
    except:
        return "Szerverhiba az adatlekérés során."

complaints_list = []

for idx, row in df_stops.iterrows():
    comp = get_complaints(row['szelesseg'], row['hosszusag'], row['nev'])
    complaints_list.append(comp)
    time.sleep(1)

df_stops['emberek_panaszai'] = complaints_list

In [ ]:
df_stops.to_excel('debrecen_buszmegallok_tisztitott.xlsx', index=False)
display(df_stops)